In [1]:
# Configurações e imports para avaliação de ranking com cross-encoders
import json
import random
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Limit dataset

In [2]:
# Carrega e normaliza o dataset
with open("limit_dataset_pt.json", "r", encoding="utf-8") as f:
    raw_dataset = json.load(f)
raw_dataset

[{'passages': {'is_selected': [1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0],
   'passage_text': ['Ácidos nucleicos, que incluem o DNA (ácido desoxirribonucleico) e o RNA (ácido ribonucleico), são feitos de monômeros conhecidos como nucleotídeos. Cada nucleotídeo tem três componentes: um açúcar de 5 carbonos, um grupo fosfato e uma base nitrogenada. Se o açúcar é desoxirribose, o polímero é DNA. Se o açúcar é ribose, o polímero é RNA. Quando o açúcar e uma base nitrogenada se combinam, formam um nucleotídeo. Nucleotídeos também são conhecidos como nucleotídeos fosfato. Ácidos nucleicos estão entre os mais importantes macromoléculas biológicas (outros sendo aminoácidos/proteínas, açúcares/carboidratos e lipídios/gorduras). Se o açúcar é ribose, o polímero é RNA. Quando o açúcar e uma base nitrogenada se combinam, formam um nucleotídeo. Nucleotídeos também são conhecidos como nucleotídeos fosfato. Á

In [3]:


# Formato esperado aqui: lista de amostras, cada uma com passages + queries
if isinstance(raw_dataset, dict) and "passages" in raw_dataset and "queries" in raw_dataset:
    dataset_samples = [raw_dataset]
elif isinstance(raw_dataset, list) and len(raw_dataset) > 0 and isinstance(raw_dataset[0], dict) and "passages" in raw_dataset[0] and "queries" in raw_dataset[0]:
    dataset_samples = raw_dataset
else:
    raise ValueError("Formato de dataset inesperado em limit_dataset_pt.json")

# Flatten para avaliação: cada query carrega as passagens da própria amostra
eval_items = []
for sample_id, sample in enumerate(dataset_samples):
    sample_passages = sample["passages"]
    sample_queries = sample["queries"]

    for q in sample_queries:
        eval_items.append({
            "sample_id": sample_id,
            "new_query": q.get("new_query", ""),
            "query_type": q.get("query_type", "desconhecido"),
            "is_selected": q.get("is_selected", []),
            "passages": sample_passages,
        })

print(f"Amostras no dataset: {len(dataset_samples)}")
print(f"Queries totais para avaliação: {len(eval_items)}")

Amostras no dataset: 2
Queries totais para avaliação: 10


In [4]:
eval_items[0]

{'sample_id': 0,
 'new_query': 'Usando apenas as passagens fornecidas, liste TODAS as evidências e condições (em conjunto, sem omitir nenhuma) que justificam o argumento central do texto: para cada evidência, cite exatamente qual requisito/fator ela sustenta e como ela se conecta ao argumento. A resposta deve cobrir todas as evidências distribuídas nas passagens; se qualquer evidência ou condição relevante estiver faltando, a resposta estará incompleta.',
 'query_type': 'completude',
 'is_selected': [0, 0],
 'passages': {'is_selected': [1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0],
  'passage_text': ['Ácidos nucleicos, que incluem o DNA (ácido desoxirribonucleico) e o RNA (ácido ribonucleico), são feitos de monômeros conhecidos como nucleotídeos. Cada nucleotídeo tem três componentes: um açúcar de 5 carbonos, um grupo fosfato e uma base nitrogenada. Se o açúcar é desoxirribose, o polímero é DNA. Se o açúcar é ribos

In [5]:
len(eval_items[0]['is_selected'])

2

## Load Corpus

In [6]:
negative_corpus = []

In [7]:
#corpus of easy negatives
#source 1: wikipedia

wikipedia_corpus = []
import datasets
wiki_dataset = datasets.load_dataset("dominguesm/wikipedia-ptbr-20230601", split="train")
for item in wiki_dataset:
    wikipedia_corpus.append(item['text'])
    if len(wikipedia_corpus) >= 10000:  # Limit to 1,000 entries for efficiency
        break

In [8]:
#source 2: news articles(https://www.kaggle.com/datasets/luisfcaldeira/folha-news-of-the-brazilian-newspaper-2024)
import pandas as pd
df = pd.read_csv("../FolhaArticles.csv", encoding="utf-8", sep="\t")
news_corpus = df['Content'].tolist()[:10000]  # Limit to 1,000 entries for efficiency

In [9]:
#add everything to negative corpus
negative_corpus.extend(wikipedia_corpus)
negative_corpus.extend(news_corpus)

## Collect hard negatives

In [10]:
# A tarefa é: para cada query, gerar um conjunto de negativos difíceis (hard negatives) a partir do negative_corpus, usando um modelo de similaridade (ex: Sentence-BERT) para selecionar os mais relevantes.
#os negatives tem que serem gerados a partir das resposta somente, e não a partir da query ou dos outros negativos. Vc devera fazer uso do is_selected para isso.

In [11]:
# Gera hard negatives por query usando SOMENTE as respostas selecionadas (is_selected)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

HARD_NEGATIVES_PER_QUERY = 40
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

def _get_passage_text(passage) -> str:
    """Extrai texto de uma passagem com fallback para diferentes chaves comuns."""
    if isinstance(passage, str):
        return passage.strip()
    if not isinstance(passage, dict):
        return ""

    for key in ["text", "content", "passage", "body", "document", "paragraph", "passage_text"]:
        value = passage.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""

def _extract_from_columnar_passages(passages_raw, selected_query):
    """Trata formato colunar: {'passage_text': [...], 'is_selected': [...]}"""
    if not isinstance(passages_raw, dict):
        return None

    passage_texts = passages_raw.get("passage_text")
    passage_flags = passages_raw.get("is_selected")

    if not isinstance(passage_texts, list):
        return None

    # 1) Prioriza flags específicos da query (q.is_selected) quando compatíveis
    if isinstance(selected_query, list) and len(selected_query) == len(passage_texts):
        # Se vier [0,0], [1,0], etc., trata como máscara 0/1
        if all(isinstance(x, (int, bool)) for x in selected_query):
            out = []
            for txt, flag in zip(passage_texts, selected_query):
                if bool(flag) and isinstance(txt, str) and txt.strip():
                    out.append(txt.strip())
            return out

    # 2) Fallback para flags globais em passages.is_selected
    if isinstance(passage_flags, list) and len(passage_flags) == len(passage_texts):
        if all(isinstance(x, (int, bool)) for x in passage_flags):
            out = []
            for txt, flag in zip(passage_texts, passage_flags):
                if bool(flag) and isinstance(txt, str) and txt.strip():
                    out.append(txt.strip())
            return out

    # 3) Sem seleção: retorna vazio
    return []

def _selected_answer_texts(item: dict) -> List[str]:
    """Retorna os textos das passagens-resposta marcadas em is_selected."""
    passages_raw = item.get("passages", [])
    selected = item.get("is_selected", [])

    if selected is None:
        selected = []

    # Primeiro tenta o formato colunar (é o caso deste dataset)
    columnar = _extract_from_columnar_passages(passages_raw, selected)
    if columnar is not None:
        return columnar

    # Formato clássico: passages como lista/dict de passagens
    passages_list = []
    passages_dict = {}
    if isinstance(passages_raw, dict):
        passages_dict = passages_raw
        passages_list = list(passages_raw.values())
    elif isinstance(passages_raw, list):
        passages_list = passages_raw
    else:
        passages_list = []

    # Caso mais comum: lista de índices das passagens corretas
    if isinstance(selected, list) and all(isinstance(x, int) for x in selected):
        texts = []
        for idx in selected:
            p = None
            if 0 <= idx < len(passages_list):
                p = passages_list[idx]
            elif isinstance(passages_dict, dict):
                if idx in passages_dict:
                    p = passages_dict[idx]
                elif str(idx) in passages_dict:
                    p = passages_dict[str(idx)]

            if p is not None:
                txt = _get_passage_text(p)
                if txt:
                    texts.append(txt)
        return texts

    # Fallback: lista booleana alinhada com passages
    if isinstance(selected, list) and all(isinstance(x, bool) for x in selected) and len(selected) == len(passages_list):
        texts = []
        for p, flag in zip(passages_list, selected):
            if flag:
                txt = _get_passage_text(p)
                if txt:
                    texts.append(txt)
        return texts

    # Fallback: IDs selecionados, quando passagens têm campo de id
    if isinstance(selected, list):
        selected_set = {str(x) for x in selected}
        texts = []
        for p in passages_list:
            if isinstance(p, dict) and str(p.get("id", "")) in selected_set:
                txt = _get_passage_text(p)
                if txt:
                    texts.append(txt)
        return texts

    return []

# Limpa e deduplica corpus negativo
negative_corpus_clean = []
seen = set()
for txt in negative_corpus:
    if not isinstance(txt, str):
        continue
    s = txt.strip()
    if s and s not in seen:
        seen.add(s)
        negative_corpus_clean.append(s)

if len(negative_corpus_clean) == 0:
    raise ValueError("negative_corpus está vazio após limpeza.")

print(f"negative_corpus limpo: {len(negative_corpus_clean)} documentos")
print(f"Gerando embeddings com modelo: {EMBEDDING_MODEL_NAME}")

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


negative_corpus limpo: 19687 documentos
Gerando embeddings com modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:

negative_embeddings = embedder.encode(
    negative_corpus_clean,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=64,
    )

Batches:   0%|          | 0/308 [00:00<?, ?it/s]

In [13]:
len(eval_items[0]['is_selected'])

2

In [14]:
# Atualiza o próprio eval_items com passagens + rótulos (sem criar novo objeto)
for item in tqdm(eval_items, desc="Atualizando eval_items com hard negatives"):
    answer_texts = _selected_answer_texts(item)

    # Se não houver resposta positiva, mantém estrutura mínima e segue
    if len(answer_texts) == 0:
        item["passages"] = {
            "passage_text": [],
            "is_selected": [],
        }
        item["is_selected"] = []
        continue

    answer_embeddings = embedder.encode(
        answer_texts,
        convert_to_numpy=True,
        show_progress_bar=False,
        normalize_embeddings=True,
    )

    # Similaridade: cada resposta vs corpus negativo
    sims = cosine_similarity(answer_embeddings, negative_embeddings)  # [n_answers, n_docs]
    max_sims = sims.max(axis=0)

    top_k = min(HARD_NEGATIVES_PER_QUERY, len(negative_corpus_clean))
    top_idx = np.argsort(-max_sims)[:top_k]

    hard_negative_texts = [negative_corpus_clean[int(idx)] for idx in top_idx]

    # Formato final por item: positivos + hard negatives
    new_passage_text = answer_texts + hard_negative_texts
    new_is_selected = [1] * len(answer_texts) + [0] * len(hard_negative_texts)

    # item["passages"] = {
    #     "passage_text": new_passage_text,
    #     "is_selected": new_is_selected,
    # }
    item["passages"] = new_passage_text
    # Mantém também no nível da query para compatibilidade com o restante do pipeline
    item["is_selected"] = new_is_selected

print(f"eval_items atualizado para {len(eval_items)} queries.")
print("Exemplo (eval_items[0]):")
eval_items[0] if eval_items else {}

Atualizando eval_items com hard negatives:   0%|          | 0/10 [00:00<?, ?it/s]

eval_items atualizado para 10 queries.
Exemplo (eval_items[0]):


{'sample_id': 0,
 'new_query': 'Usando apenas as passagens fornecidas, liste TODAS as evidências e condições (em conjunto, sem omitir nenhuma) que justificam o argumento central do texto: para cada evidência, cite exatamente qual requisito/fator ela sustenta e como ela se conecta ao argumento. A resposta deve cobrir todas as evidências distribuídas nas passagens; se qualquer evidência ou condição relevante estiver faltando, a resposta estará incompleta.',
 'query_type': 'completude',
 'is_selected': [1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'passages': ['Ácidos nucleicos, que incluem o DNA (ácido desoxirribonucleico) e o RNA (ácido ribonucleico), são feitos de monômeros conhecidos como nucleotídeos. Cada nucleotídeo tem três componentes: um açúcar de 5 carbonos, um grupo fosfato e uma base nitrogenada. Se o açúcar é desoxirr

In [15]:
#save new dataset with hard negatives
with open("limit_dataset_pt_with_hard_negatives.json", "w", encoding="utf-8") as f:
    json.dump(eval_items, f, ensure_ascii=False, indent=2)